# Notebook 02 — Tabela Bronze

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Ler os arquivos CSV brutos do Volume e persistí-los como tabela Delta, sem nenhuma transformação de conteúdo.

### Entrada e saída

| | |
|---|---|
| **Entrada** | `/Volumes/workspace/bronze/raw_ons/BALANCO_ENERGIA_SUBSISTEMA_*.csv` |
| **Saída** | `workspace.bronze.balanco_energia_bruto` |

### Princípio da camada Bronze

A Bronze é o **cofre de evidências** do pipeline: o dado como ele chegou, sem alteração. Se algo der errado nas camadas seguintes, sempre é possível voltar aqui e verificar o que a fonte entregou originalmente.

Por isso todas as colunas são lidas como **texto**, inclusive as numéricas. A tipagem é uma interpretação do dado, e interpretação pertence à Silver. Se a conversão se mostrar equivocada depois, o valor original continua disponível sem precisar de nova ingestão.

As únicas colunas acrescentadas são metadados de **linhagem**: de qual arquivo o registro veio e quando foi carregado.

## 1. Leitura dos CSVs com schema explícito

Duas decisões merecem explicação.

**Schema explícito em vez de `inferSchema`.** Inferir o schema obriga o Spark a fazer uma passada adicional sobre os dados só para adivinhar os tipos, e a adivinhação pode errar. Declarar o schema é mais rápido e determinístico.

**Separador `;`.** Confirmado pela inspeção feita no notebook anterior, contrariando o que a documentação do portal informa.

Os metadados `_arquivo_origem` e `_data_ingestao` registram a procedência de cada linha — é o que permite, mais adiante, verificar a completude arquivo por arquivo.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

VOLUME_BRONZE = "/Volumes/workspace/bronze/raw_ons"
TABELA_BRONZE = "workspace.bronze.balanco_energia_bruto"

# Na Bronze lemos TUDO como texto, de propósito: a camada é o "cofre de
# evidências", o dado exatamente como chegou. A tipagem acontece na Silver.
schema_bronze = StructType([
    StructField("id_subsistema",     StringType(), True),
    StructField("nom_subsistema",    StringType(), True),
    StructField("din_instante",      StringType(), True),
    StructField("val_gerhidraulica", StringType(), True),
    StructField("val_gertermica",    StringType(), True),
    StructField("val_gereolica",     StringType(), True),
    StructField("val_gersolar",      StringType(), True),
    StructField("val_carga",         StringType(), True),
    StructField("val_intercambio",   StringType(), True),
])

df_bronze = (
    spark.read
        .option("header", True)
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .schema(schema_bronze)
        .csv(f"{VOLUME_BRONZE}/BALANCO_ENERGIA_SUBSISTEMA_*.csv")
        .withColumn("_arquivo_origem", F.col("_metadata.file_name"))
        .withColumn("_data_ingestao", F.current_timestamp())
)

print(f"Linhas lidas: {df_bronze.count():,}")
df_bronze.printSchema()

Linhas lidas: 306,840
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: string (nullable = true)
 |-- val_gerhidraulica: string (nullable = true)
 |-- val_gertermica: string (nullable = true)
 |-- val_gereolica: string (nullable = true)
 |-- val_gersolar: string (nullable = true)
 |-- val_carga: string (nullable = true)
 |-- val_intercambio: string (nullable = true)
 |-- _arquivo_origem: string (nullable = false)
 |-- _data_ingestao: timestamp (nullable = false)



## 2. Persistência como tabela Delta

O formato **Delta** é o padrão do Databricks e fornece transações ACID, controle de versão e *time travel* — ou seja, é possível consultar como a tabela estava em execuções anteriores.

O modo `overwrite` torna o notebook reexecutável: rodar de novo substitui a tabela em vez de duplicar registros.

In [0]:
(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_BRONZE))

print(f"Tabela criada: {TABELA_BRONZE}")
display(spark.table(TABELA_BRONZE).limit(10))

Tabela criada: workspace.bronze.balanco_energia_bruto


id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio,_arquivo_origem,_data_ingestao
NE,NORDESTE,2019-01-01 00:00:00,2292.41700000,873.48200000,5320.80899999,0E-8,9831.71799999,-1345.01000000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
N,NORTE,2019-01-01 00:00:00,7297.07300000,1416.71899999,142.23700000,0E-8,4888.03300000,3967.99600000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
SIN,SISTEMA INTERLIGADO NACIONAL,2019-01-01 00:00:00,41461.54200000,7826.76300000,6081.40600000,0E-8,55369.71000000,0E-8,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
SE,SUDESTE/CENTRO-OESTE,2019-01-01 00:00:00,28304.91799999,5007.99900000,0E-8,0E-8,31079.29999999,2233.61699999,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
S,SUL,2019-01-01 00:00:00,3567.13399999,528.56300000,618.36000000,0E-8,9570.66099999,-4856.60400000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
NE,NORDESTE,2019-01-01 01:00:00,2280.10800000,739.29300000,5199.35100000,0E-8,9550.74600000,-1331.99399999,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
N,NORTE,2019-01-01 01:00:00,6900.44900000,1234.97699999,150.28500000,0E-8,4741.86400000,3543.84700000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
SIN,SISTEMA INTERLIGADO NACIONAL,2019-01-01 01:00:00,41882.89100000,7507.09900000,5858.29000000,0E-8,55248.28000000,0E-8,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
SE,SUDESTE/CENTRO-OESTE,2019-01-01 01:00:00,29349.72199999,5006.13000000,0E-8,0E-8,31008.37299999,3347.47900000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z
S,SUL,2019-01-01 01:00:00,3352.61200000,526.69899999,508.65400000,0E-8,9947.29800000,-5559.33300000,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T16:19:08.818Z


## 3. Verificação de completude

A contagem de registros por arquivo é comparada ao valor esperado, calculado antes de olhar o resultado: **4 subsistemas × 24 horas × 365 dias = 35.040 registros por ano**.

Conferir o volume contra uma expectativa, em vez de apenas "ver se carregou", é o que transforma uma checagem superficial em verificação real — e foi exatamente essa comparação que revelou a existência de um quinto registro por hora, tratado no notebook seguinte.

In [0]:
display(
    spark.table(TABELA_BRONZE)
        .groupBy("_arquivo_origem")
        .agg(F.count("*").alias("qtd_linhas"))
        .orderBy("_arquivo_origem")
)

_arquivo_origem,qtd_linhas
BALANCO_ENERGIA_SUBSISTEMA_2019.csv,43800
BALANCO_ENERGIA_SUBSISTEMA_2020.csv,43920
BALANCO_ENERGIA_SUBSISTEMA_2021.csv,43800
BALANCO_ENERGIA_SUBSISTEMA_2022.csv,43800
BALANCO_ENERGIA_SUBSISTEMA_2023.csv,43800
BALANCO_ENERGIA_SUBSISTEMA_2024.csv,43920
BALANCO_ENERGIA_SUBSISTEMA_2025.csv,43800
